# ANÁLISIS Y DISEÑO DE ALGORITMOS · 2026-B
## Informe Académico Consolidado — Primera Entrega
### Proyecto Oráculo: Formalización, NP-Dureza y Optimización Heurística

**Subproyecto:** ADAG1 — Proyecto Oráculo  
**Grupo:** GP01  
**Integrantes:** Julián Gómez Ortiz  
**Repositorio:** `https://github.com/JulianGomezOrtiz/ADA-2026-2-GP01`  
**Rama:** `main`  

---

# 1. Planteamiento y Formalización del Problema

## 1.1. Contexto y Descripción General
El **Proyecto Oráculo** aborda el problema de optimización combinatoria consistente en diseñar de forma guiada una configuración óptima de directivas de prompting y temperatura para maximizar la capacidad de un modelo de lenguaje autorregresivo (LLM) de satisfacer un conjunto heterogéneo y estricto de restricciones verificables.

El sistema toma decisiones sobre cinco componentes disjuntos pero interdependientes de la plantilla de instrucciones:
1. **Rol:** Directiva de prefijo y posicionamiento actitudinal (ejecutada antes de la respuesta).
2. **Estrategia:** Directiva de proceso deliberativo o razonamiento interno previo a la salida.
3. **Formato:** Reglas de estructura sintáctica (párrafos, secciones, viñetas, Markdown).
4. **Estilo:** Restricciones de micro-texto, puntuación, mayúsculas y caracteres permitidos.
5. **Cierre:** Instrucción de sufijo agregada tras la consulta del usuario, con posibilidad de inyección de recordatorios de restricciones.

Además de estos componentes categóricos, se evalúa el parámetro de decodificación continua **temperatura** (fijado en 0.0 para decodificación determinista greedy en la fase inicial).

El recurso crítico a optimizar es la tasa de cumplimiento estricto (precisión todo-o-nada) bajo un presupuesto computacional restringido por el costo de inferencia del modelo neuronal.

## 1.2. Entradas y Tamaño del Problema
El tamaño del problema no debe reducirse a una única variable escalar, ya que intervienen múltiples dimensiones ortogonales:
- **Catálogo de opciones $\mathcal{K}$:** Conjunto de ranuras $\mathcal{R} = \{\text{rol}, \text{estrategia}, \text{formato}, \text{estilo}, \text{cierre}\}$. Cada ranura $r \in \mathcal{R}$ posee un catálogo finito de alternativas textuales $A_r = \{a_{r,0}, a_{r,1}, \dots, a_{r,k_r-1}\}$. En este proyecto, $k_r = 8$ para todas las ranuras, determinando $|\mathcal{K}| = 8^5 = 32.768$ combinaciones de directivas discretas.
- **Conjunto de instancias de evaluación $\mathcal{I}$:** Colección de $n$ problemas $i = \langle p_i, \Phi_i \rangle$, donde $p_i$ es el texto de la petición (prompt) y $\Phi_i = \{\phi_{i,1}, \phi_{i,2}, \dots, \phi_{i,m_i}\}$ es el conjunto de $m_i$ restricciones booleanas formales que debe satisfacer la respuesta.
- **Espacio de salida y horizonte de generación:** Parámetro $T = \text{max\_new\_tokens}$ (tope de 384 tokens generados) y longitud de contexto $|p_i|$.
- **Función de evaluación (Oráculo $\mathcal{O}$):** Caja negra determinista (en temperatura 0.0) o estocástica (en temperatura $> 0$) que recibe una tupla de configuración y una instancia, y evalúa si la respuesta generada por el LLM cumple simultáneamente todas las restricciones $\phi \in \Phi_i$.

# 2. Formalización de las Dos Partes del Subproyecto (ADAG1)

Siguiendo estrictamente las especificaciones del subproyecto ADAG1, la formalización matemática se divide en dos componentes conceptualmente distintos:

### Parte 1: Configuración Óptima con Oráculo Exacto
En esta vertiente, asumimos acceso a un oráculo exacto determinista $\mathcal{O}_{exact}(c, i) \in \{0, 1\}$ que en tiempo finito $O(1)$ indica si la configuración $c$ produce una respuesta que satisface formalmente la instancia $i$. El problema consiste en hallar la configuración $c^* \in \mathcal{C}$ que maximiza la suma de aciertos sobre el universo de instancias:
$$\max_{c \in \mathcal{C}} \sum_{i \in \mathcal{I}} \mathcal{O}_{exact}(c, i)$$
Esta parte corresponde a un problema de optimización combinatoria estática con funciones de evaluación lógicas deterministas. Como se demuestra formalmente en la Sección 4, esta variante es **NP-difícil** mediante reducción desde 3-SAT.

### Parte 2: Búsqueda con Consultas Limitadas o Ruidosas
En el entorno de ejecución real con un modelo neuronal, las consultas no son instantáneas ni gratuitas, y cuando interviene temperatura $\tau > 0$, el oráculo se convierte en una variable aleatoria de Bernoulli con parámetro $p(c, i) = \mathbb{P}(\mathcal{O}_{ruidoso}(c, i) = 1)$.

En este régimen, la dificultad computacional no se modela mediante reducciones polinomiales clásicas de NP-dureza, sino a través de la **complejidad de muestreo (Sample Complexity)** y teoría del aprendizaje computacional (marco PAC / cotas de concentración de Hoeffding). Para estimar la precisión media poblacional $\mu(c) = \mathbb{E}[\mathcal{O}(c, I)]$ con un error a lo sumo $\epsilon$ y un nivel de confianza $1-\delta$, la cota de Hoeffding impone que el número mínimo de consultas por configuración satisface:
$$m \ge \frac{1}{2\epsilon^2} \ln\left(\frac{2|\mathcal{C}|}{\delta}\right)$$
Esto explica matemáticamente por qué una búsqueda exhaustiva sobre 32.768 configuraciones requeriría un presupuesto computacional astronómico, justificando el empleo de métodos heurísticos como el ascenso por coordenadas.

# 3. Salidas, Función Objetivo y Restricciones

## 3.1. Salida Concreta del Algoritmo
El algoritmo debe devolver una tupla explícita de configuración:
$$c = \langle r_{rol}, r_{estrategia}, r_{formato}, r_{estilo}, r_{cierre}, \tau \rangle$$
donde $r_k \in \{0, 1, \dots, 7\}$ para cada ranura categórica y $\tau \in \{0.0, 0.3, 0.7\}$. Dicha salida se almacena y serializa en el archivo estándar `entrega.json`.

## 3.2. Función Objetivo (Evaluación Todo-o-Nada)
Sea $n = |\mathcal{I}_{busqueda}|$ el cardinal del conjunto de instancias de búsqueda ($n = 100$). Para cada instancia $i \in \mathcal{I}_{busqueda}$, el verificador lógico evalúa el conjunto de restricciones $\Phi_i = \{\phi_{i,1}, \dots, \phi_{i,m_i}\}$. La función indicadora de éxito $I(c, i)$ está definida de forma estricta como:
$$I(c, i) = \begin{cases} 1 & \text{si } \forall j \in \{1, \dots, m_i\}, \phi_{i,j}(\text{salida}(c, i)) = \text{Verdadero} \\ 0 & \text{si } \exists j \in \{1, \dots, m_i\} \text{ tal que } \phi_{i,j}(\text{salida}(c, i)) = \text{Falso} \end{cases}$$

La métrica de desempeño a maximizar es la **precisión global empírica**:
$$P(c) = \frac{1}{n} \sum_{i=1}^n I(c, i)$$
Buscamos entonces:
$$c^* = \arg\max_{c \in \mathcal{C}} P(c)$$

## 3.3. Restricciones y Supuestos Operativos
1. **Disyunción estricta de particiones:** La búsqueda heurística debe restringirse exclusivamente al lote $\mathcal{I}_{busqueda}$ (100 instancias). El lote $\mathcal{I}_{validacion}$ (87 instancias) permanece congelado para verificación independiente.
2. **Compatibilidad con la interfaz del oráculo:** Los tipos de datos en `entrega.json` deben ser enteros para los índices de ranura y flotante para la temperatura.
3. **Invarianza de arquitectura:** Las evaluaciones se conducen bajo el modelo `Qwen3-1.7B` (`qwen17b`), con longitud máxima de generación fijada en 384 tokens.

# 4. Demostración Formal de NP-Dureza

En cumplimiento de los criterios exigidos para el subproyecto ADAG1, se formula la versión de decisión del problema de configuración con oráculo exacto y se construye una reducción polinomial desde el problema canónico **3-SAT**.

## 4.1. Versión de Decisión: `CONFIG-OPT-DECISION`
- **Instancia:** Una tupla $\langle \mathcal{R}, \{A_r\}_{r \in \mathcal{R}}, \mathcal{I}, k \rangle$, donde $\mathcal{R}$ es un conjunto de ranuras de decisión, $A_r$ es el catálogo finito de directivas disponibles para la ranura $r$, $\mathcal{I} = \{i_1, \dots, i_m\}$ es un conjunto de $m$ instancias con verificadores lógicos de respuesta deterministas, y $k \in \mathbb{Z}^+$ es un umbral de cumplimiento entero ($1 \le k \le m$).
- **Pregunta:** ¿Existe una configuración $c \in \prod_{r \in \mathcal{R}} A_r$ tal que la cantidad de instancias que cumplen simultáneamente todas sus restricciones sea al menos $k$?
$$\sum_{i \in \mathcal{I}} I(c, i) \ge k$$

## 4.2. Pertenencia a la Clase NP
Para demostrar que $\text{CONFIG-OPT-DECISION} \in \text{NP}$:
1. **Certificado polinomial:** Una configuración concreta $c = (a_1, a_2, \dots, a_{|\mathcal{R}|})$, cuya longitud está acotada por $|\mathcal{R}| \cdot \max_{r, j} |a_{r,j}|$, un tamaño lineal en la descripción de la entrada.
2. **Verificador polinomial:** Dado el certificado $c$, un algoritmo ejecuta la función de verificación sobre cada una de las $m$ instancias. Para cada instancia $i$, la evaluación de los verificadores deterministas sobre las restricciones toma tiempo $O(m_i \cdot |salida|)$. Dado que $m_i$ y la longitud de la salida están acotados polinomialmente, el conteo total $\sum_{i=1}^m I(c, i)$ y la comparación con $k$ se efectúan en tiempo determinista $O(m \cdot \text{poly}(|entrada|))$.
Por consiguiente, el problema pertenece a la clase NP.

## 4.3. Reducción Polinomial desde 3-SAT
Sea una fórmula booleana arbitraria $\phi$ en forma normal conjuntiva 3-CNF con $n$ variables $x_1, x_2, \dots, x_n$ y $m$ cláusulas $C_1, C_2, \dots, C_m$, donde cada cláusula $C_r$ es la disyunción de exactamente tres literales:
$$C_r = (\ell_{r,1} \lor \ell_{r,2} \lor \ell_{r,3}), \quad \ell_{r,j} \in \{x_1, \neg x_1, \dots, x_n, \neg x_n\}$$

Construimos una instancia $\langle \mathcal{R}, \{A_r\}, \mathcal{I}, k \rangle$ de `CONFIG-OPT-DECISION` de la siguiente manera:
1. **Ranuras y catálogo de directivas:** Por cada variable booleana $x_j$ ($j = 1, \dots, n$), definimos una ranura de decisión $r_j \in \mathcal{R}$. El catálogo $A_{r_j}$ de dicha ranura cuenta con dos opciones mutuamente excluyentes:
   - Opción $0$: $a_{j,0} = \text{"asignar } x_j = \text{Verdadero"}$
   - Opción $1$: $a_{j,1} = \text{"asignar } x_j = \text{Falso"}$
   Una configuración completa $c = (v_1, v_2, \dots, v_n) \in \{0, 1\}^n$ representa una asignación de verdad total $\tau: \{x_1, \dots, x_n\} \to \{\text{V}, \text{F}\}$.
2. **Conjunto de instancias de prueba $\mathcal{I}$:** Por cada cláusula $C_r = (\ell_{r,1} \lor \ell_{r,2} \lor \ell_{r,3})$ de $\phi$, construimos una instancia $i_r \in \mathcal{I}$. El verificador de la instancia $i_r$ se diseña de modo que la respuesta generada satisface la instancia si y solo si la configuración $c$ selecciona al menos una de las directivas correspondientes a los literales de $C_r$:
   $$I(c, i_r) = 1 \iff \bigvee_{j=1}^3 (\text{directiva correspondiente a } \ell_{r,j} \text{ está activa en } c)$$
3. **Umbral de cumplimiento:** Definimos el umbral $k = m$ (exigiendo que todas las $m$ instancias sean satisfechas simultáneamente).

## 4.4. Complejidad Polinomial de la Transformación
La construcción produce exactamente $n$ ranuras con 2 directivas cada una y $m$ instancias con 3 condiciones disyuntivas. La creación de estas estructuras toma tiempo $O(n + m)$, lo cual es lineal y estrictamente polinomial respecto al tamaño de la fórmula 3-SAT original.

## 4.5. Demostración de Equivalencia en Ambos Sentidos ($\iff$)

### $(\Rightarrow)$ Si $\phi$ es satisfacible, entonces la instancia de `CONFIG-OPT-DECISION` es positiva:
Si $\phi \in \text{3-SAT}$, existe una asignación de verdad $\tau: \{x_1, \dots, x_n\} \to \{\text{V}, \text{F}\}$ tal que cada cláusula $C_r$ tiene al menos un literal verdadero bajo $\tau$.
Construimos la configuración $c_\tau = (v_1, \dots, v_n)$ fijando $v_j = 0$ si $\tau(x_j) = \text{V}$, y $v_j = 1$ si $\tau(x_j) = \text{F}$.
Para cada cláusula $C_r$, como al menos un literal $\ell_{r,j}$ es verdadero bajo $\tau$, la directiva asociada a dicho literal forma parte de $c_\tau$. En consecuencia, el verificador de la instancia $i_r$ valida la salida como correcta: $I(c_\tau, i_r) = 1$.
Por lo tanto, la cantidad total de instancias cumplidas es:
$$\sum_{r=1}^m I(c_\tau, i_r) = m \ge k$$
con lo cual la respuesta a `CONFIG-OPT-DECISION` es afirmativa (instancia positiva).

### $(\Leftarrow)$ Si la instancia de `CONFIG-OPT-DECISION` es positiva, entonces $\phi$ es satisfacible:
Supongamos que existe una configuración $c^* = (v_1^*, \dots, v_n^*)$ tal que $\sum_{r=1}^m I(c^*, i_r) \ge k = m$.
Dado que existen exactamente $m$ instancias en total y cada $I(c^*, i_r) \le 1$, la única forma de alcanzar la suma $m$ es que **cada una** de las $m$ instancias sea satisfecha: $I(c^*, i_r) = 1$ para todo $r \in \{1, \dots, m\}$.
Definimos la asignación de variables $\tau^*$ como:
$$\tau^*(x_j) = \begin{cases} \text{V} & \text{si } v_j^* = 0 \\ \text{F} & \text{si } v_j^* = 1 \end{cases}$$
Por construcción, como cada instancia $i_r$ es satisfecha, al menos una de las directivas activadas en $c^*$ satisface uno de los literales de la cláusula $C_r$. Esto implica que bajo la asignación $\tau^*$, la cláusula $C_r$ se evalúa a Verdadero. Puesto que esto ocurre para toda cláusula $r \in \{1, \dots, m\}$, todas las cláusulas de $\phi$ se satisfacen simultáneamente. En consecuencia, $\phi$ es satisfacible ($\phi \in \text{3-SAT}$).

## 4.6. Conclusión de NP-Dureza
Habiendo demostrado que `CONFIG-OPT-DECISION` pertenece a NP y que todo problema 3-SAT se reduce polinomialmente a él ($3\text{-SAT} \le_P \text{CONFIG-OPT-DECISION}$), concluimos formalmente que:
> **El problema de optimización de configuración de instrucciones con oráculo exacto es NP-difícil.**

**Aclaración metodológica esencial:** La NP-dureza demostrada ampara exclusivamente la variante combinatoria exacta (Parte 1). La búsqueda heurística con muestreo ruidoso o instancias limitadas (Parte 2) no traslada mecánicamente esta reducción, sino que se fundamenta en cotas estadísticas de complejidad de muestreo.

# 5. Descripción del Proceso de Optimización y Método de Búsqueda

## 5.1. Espacio de Alternativas y Topología del Catálogo
El espacio está formado por 5 ranuras discretas con 8 directivas cada una:
$$|\mathcal{C}| = 8_{\text{rol}} \times 8_{\text{estrategia}} \times 8_{\text{formato}} \times 8_{\text{estilo}} \times 8_{\text{cierre}} = 32.768 \text{ combinaciones (a } \tau = 0.0\text{)}$$

Cada ranura aborda una superficie ortogonal de la respuesta del modelo:
- `rol` ataca el **prefijo** de la respuesta.
- `estrategia` influye en la **longitud y deliberación**.
- `formato` condiciona la **macro-estructura** (párrafos, encabezados, viñetas).
- `estilo` gobierna los **caracteres, casing y puntuación**.
- `cierre` condiciona el **sufijo y recordatorio** de restricciones.

El diseño pedagógico del oráculo incluye una única opción limpia por ranura, mientras que las otras 7 opciones inyectan directivas dañinas o restrictivas que entran en conflicto con las restricciones de las instancias.

## 5.2. Método Heurístico: Ascenso por Coordenadas (Coordinate Ascent)
Frente a la imposibilidad de evaluar exhaustivamente las 32.768 combinaciones con llamadas neuronales (que tardarían decenas de horas en GPU T4), se seleccionó un algoritmo de **Ascenso por Coordenadas Univariado**:

```text
Algoritmo: Ascenso por Coordenadas para Prompt Optimization
Entrada: Configuración base inicial c_0 = (0, 0, 0, 0, 0, 0.0), Lote de búsqueda I_busqueda
Salida: Mejor configuración c* y precisión P*

1. P* <- Evaluar(c_0, I_busqueda); c* <- c_0
2. Para cada componente r en [rol, estrategia, formato, estilo, cierre] hacer:
3.    Para cada opcion v en {0, 1, ..., 7} hacer:
4.       Si v == c*[r] entonces continuar;
5.       c_candidato <- Modificar c* fijando r = v
6.       P_actual <- Evaluar(c_candidato, I_busqueda)
7.       Si P_actual > P* entonces:
8.          P* <- P_actual
9.          c* <- c_candidato
10. Repetir hasta convergencia o agotamiento de cuota computacional
```

### Parámetros Relevantes de la Búsqueda
- **Lote de instancias de búsqueda:** $n = 100$ instancias curadas (promedio de 2.5 restricciones por instancia).
- **Temperatura:** $\tau = 0.0$ (greedy decoding determinista para garantizar reproducibilidad exacta).
- **Presupuesto de atención por batch:** $32.000.000$ tokens$^2$.
- **Criterio de actualización:** Selección voraz (greedy) ante mejoras estrictas ($P_{nueva} > P_{actual}$).
- **Condición de parada:** Convergencia en una ronda sin mejoras o límite presupuestal de inferencia.

# 6. Resultados de la Búsqueda y Trazabilidad Empírica

Durante la campaña de optimización en Google Colab con GPU T4, se alcanzaron a ejecutar **34 evaluaciones al oráculo** antes de que la cuota de cómputo GPU asignada se agotara. A continuación se presenta el registro de progresión verificado:

| Etapa / Componente | Rango de Evaluaciones | Precisión Inicial | Mejor Precisión Observada | Salto Observado |
|:---|:---:|:---:|:---:|:---:|
| **Configuración Inicial** | Eval 1 | 9.0% | **9.0%** | Configuración base `(0, 0, 0, 0, 0)` |
| **Exploración de Rol** | Evals 2 – 8 | 9.0% | **10.0%** | Salto en `rol = 5` (+1.0%) |
| **Exploración de Estrategia** | Evals 9 – 15 | 10.0% | **13.0%** | Salto en `estrategia = 2` (+3.0%) |
| **Exploración de Formato** | Evals 16 – 22 | 13.0% | **13.0%** | Ninguna superó formato 0 (todas $\le 10\%$) |
| **Exploración de Estilo** | Evals 23 – 29 | 13.0% | **16.0%** | **Máximo en `estilo = 6` (Eval 27: +3.0%)** |
| **Exploración de Cierre** | Evals 30 – 34 | 16.0% | **16.0%** | Opciones 1, 2, 4, 5, 6 dieron $\le 14\%$ |

### Hito Principal de la Búsqueda
- La **máxima precisión registrada** fue de **16.0% (16/100 casos correctos)**, alcanzada en la **evaluación número 27**.
- Dicho resultado corresponde a la configuración:
  $$\{\text{rol}: 5, \text{estrategia}: 2, \text{formato}: 0, \text{estilo}: 6, \text{cierre}: 0, \text{temperatura}: 0.0\}$$
- La mejora neta lograda sobre la línea base inicial fue de **+7 puntos porcentuales** (de 9.0% a 16.0%), representando un incremento relativo del 77.7% en la tasa de acierto.

### Advertencia sobre Optimalidad Global
> **IMPORTANTE:** El valor de 16.0% representa el **mejor resultado empíricamente observado** bajo el presupuesto explorado, y **no una garantía matemática de optimalidad global**. Al haberse interrumpido la búsqueda en la evaluación 34 por agotamiento de cuota GPU, quedaron pendientes por explorar opciones de cierre (particularmente `cierre = 3`) y rondas adicionales de refinamiento cruzado.

# 7. Análisis de Trazas de Fallos, Descarte de Opciones e Incidencias

Una lista ciega de precisiones no satisface el rigor exigido por el curso. El examen sistemático de las trazas de fallos (`violo`, `salida`) permitió fundamentar el descarte de familias enteras de directivas:

### 1. Diagnóstico de la ranura de Rol
- **Evidencia en trazas:** Las directivas de rol con saludos o aperturas obligatorias (ej. rol 0 *"Begin every reply with 'Happy to help!'"*, rol 3 *"Start every reply with 'Answer:'"*) causaron violaciones inmediatas de la familia `start_end:starts_with` y dispararon conteos no autorizados en `length_constraints:number_of_words`.
- **Decisión de descarte:** Se descartaron todas las directivas de prefijo rígido. Se adoptó `rol = 5` (*"You are a meticulous assistant. Follow every formatting requirement exactly, and add nothing the request did not ask for."*), cuya instrucción de no agregar nada superfluo neutralizó las penalizaciones de preámbulo.

### 2. Diagnóstico de la ranura de Estrategia
- **Evidencia en trazas:** Las estrategias que forzaban cadenas de pensamiento verborrágicas (ej. estrategia 0 *"write your reasoning as a numbered list..."*, estrategia 3 *"expand every point into its own paragraph"*) inflaron el consumo de tokens y rompieron límites máximos de palabras en instancias de concisión estricta.
- **Decisión de descarte:** Se seleccionó `estrategia = 2` (*"Write a draft, check it against each requirement, then output only the corrected final version."*), la cual induce verificación interna sin emitir tokens no deseados en la respuesta final.

### 3. Diagnóstico de la ranura de Formato y la Incidencia `langdetect`
- **Incidencia técnica:** Durante la evaluación 22 (formato 7) se reportó:
  `ERROR:absl:Unable to detect language for text ... due to No features in text.`
- **Causa identificada:** La directiva de numerar obligatoriamente cada párrafo colisionó con una instancia que exigía una respuesta de una sola frase corta. El modelo colapsó generando una salida casi vacía o de caracteres de puntuación sin palabras reconocibles, lo que impidió al clasificador de idioma de IFEval extraer características léxicas.
- **Decisión de descarte:** Se descartaron los formatos invasivos (Markdown con encabezados `#` o viñetas obligatorias), reteniendo `formato = 0` (*"Output only the requested text. No preamble, no commentary, no headings."*).

### 4. Justificación del Salto a 16% mediante Estilo (`estilo = 6`)
- La directiva `estilo = 6` (*"Respect exact word counts, casing and punctuation rules. Use no character the request did not ask for."*) atacó directamente las tres causas más frecuentes de fallo registradas en las trazas: exceso de palabras, mayúsculas incorrectas y signos de puntuación parásitos. Esto explica el salto cualitativo registrado en la evaluación 27.

# 8. Validación Final y Reproducibilidad

## 8.1. Diferenciación Metodológica de Conjuntos
Es mandatorio distinguir los tres conjuntos de datos involucrados en el proyecto:
1. **Conjunto de búsqueda (100 instancias):** Utilizado durante las 34 evaluaciones para comparar alternativas y guiar el ascenso por coordenadas.
2. **Conjunto de validación interno (87 instancias):** Partición disjunta curada en `datos_visibles.json`. Se emplea mediante `oraculo.validar(config, n)` para evaluar la capacidad de generalización frente a nuevas instancias con la misma distribución de dificultad.
3. **Conjunto de prueba privado del docente:** Datos reservados y no accesibles para el equipo, sobre los cuales la profesora ejecutará `Evaluacion_Inicial.ipynb` para la calificación oficial definitiva.

## 8.2. Invocación de Validación y Coherencia
La configuración registrada en `entrega.json`:
```json
{
 "grupo": "GP01",
 "config": {
  "rol": 5,
  "estrategia": 2,
  "formato": 0,
  "estilo": 6,
  "cierre": 0,
  "temperatura": 0.0
 },
 "semana": 3
}
```
es **estrictamente coherente** entre los tres artefactos del repositorio:
- En `Documento.ipynb` se formaliza y justifica con las trazas y la tabla de 34 evaluaciones.
- En `Busqueda_Inicial.ipynb` se encuentra el código y la bitácora reproducible que la produjo en la evaluación 27.
- En `Evaluacion_Inicial.ipynb` es leída directamente por el docente sin intervención manual para su evaluación ciega.

# 9. Organización del Repositorio y Manual Técnico

El repositorio del Grupo 1 (`ADAG1 - GP01`) se encuentra estructurado de acuerdo con las especificaciones de entrega:

```text
ADA-2026-2-GP01/
├── README.md                 # Guía general de instalación, dependencias y uso rápido
├── MANUAL_TECNICO.md         # Manual técnico detallado (entorno, arquitectura, flujos, pruebas)
├── Documento.ipynb           # Informe académico consolidado (formalización, NP-dureza, resultados)
├── Busqueda_Inicial.ipynb    # Bitácora experimental completa (34 evaluaciones, método, trazas)
├── Evaluacion_Inicial.ipynb  # Notebook oficial de evaluación automática para la profesora
├── entrega.json              # Configuración ganadora oficial serializada
├── oraculo.py                # Módulo de caja negra del oráculo y verificadores IFEval
├── ayudas.py                 # Funciones de soporte (carga de modelo, particionado, visualización)
└── datos_visibles.json       # 450 instancias curadas (100 búsqueda, 87 validación, 263 descartadas)
```

### Verificación de Cumplimiento de Criterios de la Guía
- **Informe de formalización y NP-dureza:** Desarrollado exhaustivamente en las Secciones 1, 2, 3 y 4 con reducción desde 3-SAT y análisis PAC.
- **Código ejecutable y reproducible:** Implementado en `Busqueda_Inicial.ipynb` y `Evaluacion_Inicial.ipynb`.
- **Manual técnico independiente:** Accesible directamente desde `MANUAL_TECNICO.md` y documentado en el `README.md`.
- **Configuración serializada válida:** Generada y verificada en `entrega.json`.